# TREV Gradient Benchmark: Autograd vs Parameter-Shift

Compare two gradient methods for tensor ring VQE:
1. **Parameter-shift**: 2P circuit evaluations (standard, exact)
2. **Autograd (vectorized)**: 1 forward + 1 backward with T-batched Kronecker contraction (exact, new)

Measures **accuracy** (vs finite-difference reference) and **wall-clock time** across:
- HEA circuits (varying N, chi, layers)
- QAOA circuits
- Different Hamiltonians (MaxCut ring, chain)

In [ ]:
# Install TREV (run once)
!pip install -q git+https://github.com/keunjunpark/TREV.git@real_form_autograd

In [ ]:
import torch
import time
import numpy as np
import pandas as pd

from TREV.circuit import Circuit
from TREV.hamiltonian.hamiltonian import Hamiltonian
from TREV.measure.enums import MeasureMethod
from TREV.measure.efficient_contraction import expectation_value_batch as ev_exact
from TREV.optimization.gradients.autograd_gradient import (
    AutogradGradient, autograd_gradient, _auto_term_chunk,
)
from TREV.optimization.gradients.batch_parameter_shift import (
    BatchParameterShiftGradient, batch_gradient,
)
from TREV.optimization.optimizer import Optimizer
from TREV.optimization.optimization import minimize

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    free_b, total_b = torch.cuda.mem_get_info(device)
    print(f'GPU memory: {free_b/1e9:.1f} GB free / {total_b/1e9:.1f} GB total')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Helpers ──

def build_maxcut_ham(N, topology='ring'):
    """MaxCut Hamiltonian: H = sum 0.5*(I - ZiZj) over edges."""
    h = Hamiltonian(num_qubits=N)
    edges = [(i, (i+1) % N) for i in range(N)] if topology == 'ring' \
            else [(i, i+1) for i in range(N-1)]
    for i, j in edges:
        h.add_pauli('I' * N, 0.5)
        p = ['I'] * N; p[i] = 'Z'; p[j] = 'Z'
        h.add_pauli(''.join(p), -0.5)
    return h

def build_hea(N, chi, L, topology='ring'):
    """Hardware-efficient ansatz: H + [CNOT ring/chain + RY,RZ] x L."""
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    for _ in range(L):
        if topology == 'ring':
            for i in range(N): c.cx(i, (i+1) % N)
        else:
            for i in range(N-1): c.cx(i, i+1)
        for i in range(N): c.ry(i); c.rz(i)
    return c

def build_qaoa(N, chi, p, topology='ring'):
    """QAOA ansatz: H + [CNOT-RZ-CNOT per edge + RX] x p."""
    c = Circuit(num_qubit=N, rank=chi, device=device)
    for i in range(N): c.h(i)
    edges = [(i, (i+1) % N) for i in range(N)] if topology == 'ring' \
            else [(i, i+1) for i in range(N-1)]
    for _ in range(p):
        for i, j in edges:
            c.cx(i, j); c.rz(j); c.cx(i, j)
        for i in range(N): c.rx(i)
    return c

def fd_grad(theta, circuit, h, eps=1e-4):
    """Finite-difference gradient (reference)."""
    g = torch.zeros(theta.numel(), device=device)
    for k in range(theta.numel()):
        tp = theta.clone(); tp[k] += eps
        fp = ev_exact(circuit.build_tensor(tp), h, device=device).item()
        tp[k] -= 2 * eps
        fm = ev_exact(circuit.build_tensor(tp), h, device=device).item()
        g[k] = (fp - fm) / (2 * eps)
    return g

def ps_grad(theta, circuit, h):
    """Parameter-shift gradient."""
    return batch_gradient(theta, circuit, h, 8, 0, np.pi/2, 1, 0, False,
                          MeasureMethod.EFFICIENT_CONTRACTION)

def cos_sim(a, b):
    return torch.nn.functional.cosine_similarity(
        a.unsqueeze(0), b.unsqueeze(0)
    ).item()

def bench(fn, warmup=3, repeats=5):
    for _ in range(warmup): fn()
    if device == 'cuda': torch.cuda.synchronize()
    times = []
    for _ in range(repeats):
        if device == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter()
        fn()
        if device == 'cuda': torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

print('Helpers loaded.')

## 1. Accuracy: HEA circuits (varying N, chi, layers)

Compare autograd gradient directly against parameter-shift (analytically exact).
Finite-difference is shown as a sanity check but is itself approximate (eps=1e-4 breaks down for deep circuits).

In [ ]:
torch.manual_seed(42)
rows = []

hea_configs = [
    (8, 4, 1), (8, 4, 2), (8, 10, 2), (8, 10, 3),
    (12, 4, 2), (12, 10, 2),
    (16, 4, 2), (16, 10, 2),
]

for N, chi, L in hea_configs:
    c = build_hea(N, chi, L)
    h = build_maxcut_ham(N)
    theta = torch.randn(c.params_size, device=device)

    ga = autograd_gradient(theta, c, h, torch.cfloat)
    gp = ps_grad(theta, c, h)

    cos_ap = cos_sim(ga, gp)
    maxerr_ap = (ga - gp).abs().max().item()
    relerr = maxerr_ap / gp.abs().max().item() if gp.abs().max().item() > 0 else 0

    rows.append({
        'circuit': 'HEA', 'N': N, 'chi': chi, 'L': L,
        'P': c.params_size, 'T': len(h.paulis),
        'cos(ad,ps)': cos_ap,
        'max|ad-ps|': maxerr_ap,
        'rel_err': relerr,
        'status': 'PASS' if cos_ap > 0.9999 else 'FAIL',
    })
    print(f"N={N:>2} chi={chi:>2} L={L} P={c.params_size:>3}  "
          f"cos(ad,ps)={cos_ap:.6f}  max|diff|={maxerr_ap:.2e}  rel={relerr:.2e}  "
          f"{'PASS' if cos_ap > 0.9999 else 'FAIL'}")

df_acc_hea = pd.DataFrame(rows)
df_acc_hea

## 2. Accuracy: QAOA circuits

Autograd vs parameter-shift (both analytically exact).

In [ ]:
torch.manual_seed(42)
rows = []

qaoa_configs = [
    (8, 4, 2), (8, 10, 2), (8, 10, 3),
    (12, 4, 2), (12, 10, 2),
    (16, 4, 2),
]

for N, chi, p in qaoa_configs:
    c = build_qaoa(N, chi, p)
    h = build_maxcut_ham(N)
    theta = torch.randn(c.params_size, device=device)

    ga = autograd_gradient(theta, c, h, torch.cfloat)
    gp = ps_grad(theta, c, h)

    cos_ap = cos_sim(ga, gp)
    maxerr_ap = (ga - gp).abs().max().item()
    relerr = maxerr_ap / gp.abs().max().item() if gp.abs().max().item() > 0 else 0

    rows.append({
        'circuit': 'QAOA', 'N': N, 'chi': chi, 'p': p,
        'P': c.params_size, 'T': len(h.paulis),
        'cos(ad,ps)': cos_ap,
        'max|ad-ps|': maxerr_ap,
        'rel_err': relerr,
        'status': 'PASS' if cos_ap > 0.9999 else 'FAIL',
    })
    print(f"N={N:>2} chi={chi:>2} p={p} P={c.params_size:>3}  "
          f"cos(ad,ps)={cos_ap:.6f}  max|diff|={maxerr_ap:.2e}  rel={relerr:.2e}  "
          f"{'PASS' if cos_ap > 0.9999 else 'FAIL'}")

df_acc_qaoa = pd.DataFrame(rows)
df_acc_qaoa

## 3. Speed: HEA scaling (N, chi, layers)

Time a single gradient call. Speedup > 1 means autograd is faster.

In [ ]:
torch.manual_seed(42)
rows = []

speed_configs = [
    # (N, chi, L)
    (4, 4, 2),
    (8, 4, 2), (8, 10, 2), (8, 10, 3),
    (12, 4, 2), (12, 10, 2),
    (16, 4, 2), (16, 10, 2),
    (20, 4, 2), (20, 10, 2),
    (24, 4, 2),
]

print(f"{'config':<28} {'P':>4} {'T':>4} | {'autograd':>10} {'param-shift':>12} {'speedup':>8}")
print("-" * 75)

for N, chi, L in speed_configs:
    torch.cuda.empty_cache()
    c = build_hea(N, chi, L)
    h = build_maxcut_ham(N)
    theta = torch.randn(c.params_size, device=device)
    P = c.params_size
    T = len(h.paulis)
    tc = _auto_term_chunk(N, chi, torch.cfloat, device)

    try:
        ms_ad = bench(lambda: autograd_gradient(theta, c, h, torch.cfloat))
        ms_ps = bench(lambda: ps_grad(theta, c, h))
        sp = ms_ps / ms_ad

        label = f"HEA N={N:>2} chi={chi:>2} L={L}"
        print(f"{label:<28} {P:>4} {T:>4} | {ms_ad:>8.1f}ms {ms_ps:>10.1f}ms {sp:>7.2f}x")

        rows.append({
            'circuit': 'HEA', 'N': N, 'chi': chi, 'L': L,
            'P': P, 'T': T, 'term_chunk': min(tc, T),
            'autograd_ms': ms_ad, 'param_shift_ms': ms_ps,
            'speedup': sp,
        })
    except Exception as e:
        print(f"HEA N={N:>2} chi={chi:>2} L={L}  error: {e}")

df_speed_hea = pd.DataFrame(rows)
df_speed_hea

## 4. Speed: QAOA scaling

In [ ]:
torch.manual_seed(42)
rows = []

qaoa_speed = [
    (8, 4, 2), (8, 10, 2), (8, 10, 3),
    (12, 4, 2), (12, 10, 2),
    (16, 4, 2), (16, 10, 2),
    (20, 4, 2),
]

print(f"{'config':<28} {'P':>4} {'T':>4} | {'autograd':>10} {'param-shift':>12} {'speedup':>8}")
print("-" * 75)

for N, chi, p in qaoa_speed:
    torch.cuda.empty_cache()
    c = build_qaoa(N, chi, p)
    h = build_maxcut_ham(N)
    theta = torch.randn(c.params_size, device=device)
    P = c.params_size
    T = len(h.paulis)

    try:
        ms_ad = bench(lambda: autograd_gradient(theta, c, h, torch.cfloat))
        ms_ps = bench(lambda: ps_grad(theta, c, h))
        sp = ms_ps / ms_ad

        label = f"QAOA N={N:>2} chi={chi:>2} p={p}"
        print(f"{label:<28} {P:>4} {T:>4} | {ms_ad:>8.1f}ms {ms_ps:>10.1f}ms {sp:>7.2f}x")

        rows.append({
            'circuit': 'QAOA', 'N': N, 'chi': chi, 'p': p,
            'P': P, 'T': T,
            'autograd_ms': ms_ad, 'param_shift_ms': ms_ps,
            'speedup': sp,
        })
    except Exception as e:
        print(f"QAOA N={N:>2} chi={chi:>2} p={p}  error: {e}")

df_speed_qaoa = pd.DataFrame(rows)
df_speed_qaoa

## 5. Speed plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# -- HEA plot --
ax = axes[0]
for chi_val in df_speed_hea['chi'].unique():
    sub = df_speed_hea[df_speed_hea['chi'] == chi_val]
    ax.plot(sub['N'], sub['autograd_ms'], 'o-', label=f'Autograd chi={chi_val}')
    ax.plot(sub['N'], sub['param_shift_ms'], 's--', label=f'Param-shift chi={chi_val}', alpha=0.7)
ax.set_xlabel('N (qubits)')
ax.set_ylabel('Time (ms)')
ax.set_title('HEA (L=2): Gradient Time')
ax.legend(fontsize=8)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

# -- QAOA plot --
ax = axes[1]
for chi_val in df_speed_qaoa['chi'].unique():
    sub = df_speed_qaoa[df_speed_qaoa['chi'] == chi_val]
    ax.plot(sub['N'], sub['autograd_ms'], 'o-', label=f'Autograd chi={chi_val}')
    ax.plot(sub['N'], sub['param_shift_ms'], 's--', label=f'Param-shift chi={chi_val}', alpha=0.7)
ax.set_xlabel('N (qubits)')
ax.set_ylabel('Time (ms)')
ax.set_title('QAOA (p=2): Gradient Time')
ax.legend(fontsize=8)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Speedup bar chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# HEA speedup
ax = axes[0]
labels = [f"N={r['N']} chi={r['chi']}" for _, r in df_speed_hea.iterrows()]
vals = df_speed_hea['speedup'].values
colors = ['#2ecc71' if v >= 1 else '#e74c3c' for v in vals]
bars = ax.bar(range(len(vals)), vals, color=colors)
ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Speedup (autograd / param-shift)')
ax.set_title('HEA: Autograd Speedup')
ax.grid(True, alpha=0.3, axis='y')

# QAOA speedup
ax = axes[1]
labels = [f"N={r['N']} chi={r['chi']}" for _, r in df_speed_qaoa.iterrows()]
vals = df_speed_qaoa['speedup'].values
colors = ['#2ecc71' if v >= 1 else '#e74c3c' for v in vals]
bars = ax.bar(range(len(vals)), vals, color=colors)
ax.axhline(y=1, color='black', linestyle='--', alpha=0.5)
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Speedup (autograd / param-shift)')
ax.set_title('QAOA: Autograd Speedup')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 7. VQE Convergence: Autograd vs Parameter-Shift

Run full optimization and compare convergence curves and wall-clock time.

In [ ]:
torch.manual_seed(42)

vqe_configs = [
    ('HEA', 12, 10, 2),
    ('HEA', 16, 4, 2),
    ('HEA', 20, 4, 2),
]
iters = 50

fig, axes = plt.subplots(1, len(vqe_configs), figsize=(6 * len(vqe_configs), 5))
if len(vqe_configs) == 1:
    axes = [axes]

for idx, (ctype, N, chi, L) in enumerate(vqe_configs):
    torch.cuda.empty_cache()
    c = build_hea(N, chi, L)
    h = build_maxcut_ham(N)
    theta = torch.randn(c.params_size, device=device)
    opt = Optimizer(torch.optim.Adam, {'lr': 0.05})

    # Autograd
    grad_ad = AutogradGradient()
    grad_ad._verbose = False
    t0 = time.time()
    _, ev_ad, _, times_ad = minimize(c, theta.clone(), h, opt, grad_ad,
                                      iteration=iters, best_value_method='contraction')
    t_ad = time.time() - t0

    # Param-shift
    grad_ps = BatchParameterShiftGradient(
        shift=np.pi/2, batch_size=None, shots=0,
        measure_method=MeasureMethod.EFFICIENT_CONTRACTION, depth=1,
    )
    grad_ps._verbose = False
    t0 = time.time()
    _, ev_ps, _, times_ps = minimize(c, theta.clone(), h, opt, grad_ps,
                                      iteration=iters, best_value_method='contraction')
    t_ps = time.time() - t0

    ev_ad = [v.item() if hasattr(v, 'item') else float(v) for v in ev_ad]
    ev_ps = [v.item() if hasattr(v, 'item') else float(v) for v in ev_ps]

    ax = axes[idx]
    ax.plot(ev_ad, label=f'Autograd ({t_ad:.1f}s)', linewidth=2)
    ax.plot(ev_ps, label=f'Param-shift ({t_ps:.1f}s)', linewidth=2, linestyle='--')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Energy')
    ax.set_title(f'{ctype} N={N} chi={chi} L={L}\nP={c.params_size} (speedup {t_ps/t_ad:.2f}x)')
    ax.legend()
    ax.grid(True, alpha=0.3)

    print(f"{ctype} N={N} chi={chi} L={L}: "
          f"autograd {t_ad:.1f}s ({t_ad/iters*1000:.0f}ms/iter), "
          f"param-shift {t_ps:.1f}s ({t_ps/iters*1000:.0f}ms/iter), "
          f"speedup {t_ps/t_ad:.2f}x")

plt.tight_layout()
plt.show()

## 8. GPU Memory: Auto term_chunk

Shows how the vectorized autograd auto-selects term batching based on GPU memory.

In [ ]:
rows = []
mem_configs = [
    (8, 4, 2), (8, 10, 2), (8, 32, 2),
    (12, 4, 2), (12, 10, 2), (12, 32, 2),
    (16, 4, 2), (16, 10, 2),
    (20, 4, 2), (20, 10, 2),
]

print(f"{'config':<25} {'T':>4} {'auto_chunk':>11} {'mem/term':>10} {'peak(MB)':>10} {'time(ms)':>10}")
print("-" * 75)

for N, chi, L in mem_configs:
    torch.cuda.empty_cache()
    c = build_hea(N, chi, L)
    h = build_maxcut_ham(N)
    T = len(h.paulis)
    theta = torch.randn(c.params_size, device=device)
    tc = _auto_term_chunk(N, chi, torch.cfloat, device)

    elem_size = 8
    mem_per_term = 6 * N * (chi ** 4) * elem_size

    try:
        # Warmup
        autograd_gradient(theta, c, h, torch.cfloat)
        torch.cuda.synchronize()

        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)

        ms = bench(lambda: autograd_gradient(theta, c, h, torch.cfloat), warmup=2, repeats=3)
        peak = torch.cuda.max_memory_allocated(device) / 1e6

        label = f"N={N:>2} chi={chi:>2} L={L}"
        print(f"{label:<25} {T:>4} {min(tc,T):>11} {mem_per_term/1e6:>8.1f}MB {peak:>8.1f}MB {ms:>8.1f}ms")
        rows.append({
            'N': N, 'chi': chi, 'L': L, 'T': T,
            'auto_chunk': min(tc, T), 'mem_per_term_MB': mem_per_term/1e6,
            'peak_MB': peak, 'time_ms': ms,
        })
    except Exception as e:
        print(f"{label:<25} {T:>4}  error: {str(e)[:50]}")

pd.DataFrame(rows)